# Kafka Demo

## Important: Connect to Kafka Broker Server FIRST

**Before running any code in this notebook**, establish an SSH tunnel to the Kafka server:

```
ssh -L <local_port>:localhost:<remote_port> <user>@<remote_server> -NTf
```

Find connection details (remote_server, user, password, ports) on the Canvas lab 1 assignment page.

**Verify your connection is active:**
```bash
lsof -i :<local_port>  # Should show an ssh process
```

**To kill the connection when done:**
```
lsof -ti:<local_port> | xargs kill -9
```

---

## Setup

It is recommended to set up a python environment for this lab (and all other ones).
```
python -m venv <environment_name>
source <environment_name>/bin/activate  # On Windows: <environment_name>\Scripts\activate
```

Then install the requirements:
```
pip install -r requirements.txt
```
Or manually:
```
pip install kafka-python
```

In [7]:
import os
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer, TopicPartition
from typing import Dict, Any

# [TODO]: Fill in a unique identifier so your topic doesn't collide with others'
# Replace ... with your andrew_id as a string (e.g., "asmith") or any unique identifier
andrew_id = "autumnq"  # Example: andrew_id = "asmith"
topic = f"lab01-{andrew_id}"
print(f"Topic: {topic}")

Topic: lab01-autumnq


### Producer Mode -> Writes Data to Broker

In [8]:
# Below schema is for messages. You may change the city data if you wish but it is optional.
def make_city_data(city: str, temperature_f: str) -> Dict[str, Any]:
    return {
        "city": city,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "temperature_f": temperature_f, # temperature in fahrenheit
    }

In [10]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html

# [TODO]: Fill in the address of your Kafka bootstrap server
# [TODO]: Kafka expects messages as bytes. Explore the documentation and decide how to serialize Python dict objects into bytes.
# Hint: You may want to convert your Python dict → JSON string → UTF-8 bytes.

producer = KafkaProducer(bootstrap_servers=["localhost:9092"],
                        value_serializer=lambda v: dumps(v).encode("utf-8"))

# bootstrap_servers tells the producer where to make its first connection so it can discover the Kafka cluster

In [11]:
# [TODO]: Add a few more cities below, as (city, temperature_f) pairs
cities = [("Pittsburgh", 64), ("NYC", 72), ("LA", 80)]

# 20 messages at 0.5s takes the same ~10 seconds the original 10-at-1s did, but gives
# you a wider range of offsets to read back from later.
NUM_MESSAGES = 20

print("Writing to Kafka Broker")
assigned_offsets = []
for i in range(NUM_MESSAGES):
    city, temperature_f = cities[randint(0, len(cities) - 1)]  # random selection
    data = make_city_data(city, temperature_f)
    future = producer.send(topic=topic, value=data)
    # Kafka assigns every message an offset: its position in the partition's log.
    assigned_offsets.append(future.get(timeout=10).offset)
    sleep(0.5)

producer.flush()
print(f"Data written to topic: {topic}")
print(f"Offsets assigned in this run: {assigned_offsets[0]} .. {assigned_offsets[-1]}")

Writing to Kafka Broker
Data written to topic: lab01-autumnq
Offsets assigned in this run: 0 .. 19


### Consumer Mode -> Reads Data from Broker

In [15]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html

# [TODO]: Fill in the missing parameters:
#   1. First parameter: topic name (should match the topic you used in producer)
#   2. bootstrap_servers: same address you used in producer (e.g., ['localhost:9092'])
#   3. auto_offset_reset: try 'earliest' to read from beginning, 'latest' for new messages only
# Note: Since producer uses value_serializer, message.value is bytes. We decode and parse JSON.
# Note: auto_offset_reset only chooses between the two ENDS of the log, and only when this
#       consumer has no valid committed offset. The next section shows how to start anywhere
#       in between. Stop this cell with the interrupt button once you have seen some messages.

consumer = KafkaConsumer(
    topic,  # [TODO]: Use your topic variable here
    bootstrap_servers=["localhost:9092"],  # [TODO]: Same bootstrap server as producer
    auto_offset_reset='latest',  # [TODO]: Try 'earliest', 'latest', or 'none'
    # Commit that an offset has been read
    enable_auto_commit=True,
    # How often to tell Kafka, an offset has been read
    auto_commit_interval_ms=1000
)

print('Reading Kafka Broker')
for message in consumer:
    # Producer serialized to JSON bytes, so we decode and parse
    message_str = message.value.decode('utf-8')
    message_dict = loads(message_str)
    print(f"offset {message.offset}: {message_dict}")
    os.system(f"echo {message_str} >> kafka_log.csv")

Reading Kafka Broker


KeyboardInterrupt: 

### Reading From the Middle of the Log

`auto_offset_reset` only picks between the two *ends* of the log -- `earliest` (the log
start offset) and `latest` (the high watermark) -- and Kafka only applies it when a
consumer has no valid committed offset. It cannot start you anywhere in between.

Real consumers need that middle ground: "replay the last 50 messages", "resume 200
messages before where we crashed". For those you use `seek()`.

The cell below does this in one pass: find the two ends of the valid range, then read
from a few different offsets spread across it.

In [18]:
# We use assign() rather than subscribe() here: it hands us a specific partition
# immediately, so seek() works without waiting for a consumer-group rebalance.

# [TODO]: Fill in the bootstrap server address (same as the producer)
explorer = KafkaConsumer(
    bootstrap_servers=["localhost:9092"],
    enable_auto_commit=False,   # don't move any committed offset while exploring
    group_id=None,              # not part of a group: nothing is committed
)
tp = TopicPartition(topic, 0)   # our lab topic has a single partition
explorer.assign([tp])

# [TODO]: Look up the two ends of the valid range.
# Hint: KafkaConsumer has beginning_offsets() and end_offsets(). Both take a LIST of
#       TopicPartition and return a dict keyed by TopicPartition.
first_offset = explorer.beginning_offsets([tp])[tp]  # low watermark: the offset of the FIRST message in the log
next_offset = explorer.end_offsets([tp])[tp]  # high watermark: the offset the NEXT produced message will get
print(f"readable offsets: {first_offset} .. {next_offset - 1}\n")


def read_from(offset, n=2):
    """Seek to `offset`, then print the next `n` messages. Provided for you."""
    explorer.seek(tp, offset)
    seen = 0
    while seen < n:
        batch = explorer.poll(timeout_ms=2000, max_records=n - seen)
        if not batch:
            break
        for record in batch[tp]:
            print(f"  offset {record.offset}: {loads(record.value.decode('utf-8'))}")
            seen += 1


# [TODO]: Pick THREE start offsets spread across the valid range -- not just the ends.
# Compute them from first_offset / next_offset so they work for any log length.
# Suggestion: the first offset, the midpoint, and 2 before the high watermark
# (a "give me the last 2 messages" replay).
start_offsets = [first_offset, (first_offset + next_offset) // 2, next_offset - 2]

for start in start_offsets:
    print(f"--- seek to offset {start} ---")
    read_from(start, 2)

explorer.close()

# Be ready to tell the TA which of these three you could have reached with
# auto_offset_reset alone, and which one needed seek().

readable offsets: 0 .. 19

--- seek to offset 0 ---
  offset 0: {'city': 'NYC', 'timestamp': '2026-08-27 20:16:51', 'temperature_f': 72}
  offset 1: {'city': 'Pittsburgh', 'timestamp': '2026-08-27 20:16:52', 'temperature_f': 64}
--- seek to offset 10 ---
  offset 10: {'city': 'Pittsburgh', 'timestamp': '2026-08-27 20:16:56', 'temperature_f': 64}
  offset 11: {'city': 'Pittsburgh', 'timestamp': '2026-08-27 20:16:57', 'temperature_f': 64}
--- seek to offset 18 ---
  offset 18: {'city': 'Pittsburgh', 'timestamp': '2026-08-27 20:17:03', 'temperature_f': 64}
  offset 19: {'city': 'NYC', 'timestamp': '2026-08-27 20:17:04', 'temperature_f': 72}


# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat


Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html

In [ ]:
# [TODO] 1. Consume the first 5 messages from your topic, showing the offset of each:
# kcat -b localhost:<local_port> -t <your_topic_name> -C -o beginning -c 5 -f "%o: %s\n"
#
# Where:
# - -b: broker address (same port as your SSH tunnel)
# - -t: your topic name (e.g., "lab01-asmith")
# - -C: consumer mode
# - -o beginning: start from the earliest offset
# - -c 5: consume 5 messages
# - -f "%o: %s\n": format to show offset and message
#
# [TODO] 2. Run it again starting from an absolute offset in the MIDDLE of your range
#           (e.g. -o 8 instead of -o beginning) and confirm the offsets line up with
#           what the Python consumer showed you.
#
# -o also accepts other forms you may find useful in the group project:
#   -o -<N>    relative to the end, e.g. -o -5 for the last 5 messages
#   -o s@<ms>  the first message at or after a timestamp in milliseconds
#
# Paste your output below and explain what the offset refers to.

### Your kcat output

kcat -b localhost:9092 -t lab01-autumnq -C -o beginning -c 5 -f "%o: %s\n"

0: {"city": "NYC", "timestamp": "2026-08-27 20:16:51", "temperature_f": 72}
1: {"city": "Pittsburgh", "timestamp": "2026-08-27 20:16:52", "temperature_f": 64}
2: {"city": "LA", "timestamp": "2026-08-27 20:16:52", "temperature_f": 80}
3: {"city": "LA", "timestamp": "2026-08-27 20:16:53", "temperature_f": 80}
4: {"city": "NYC", "timestamp": "2026-08-27 20:16:53", "temperature_f": 72}

kcat -b localhost:9092 -t lab01-autumnq -C -o 8 -c 5 -f "%o: %s\n"
8: {"city": "Pittsburgh", "timestamp": "2026-08-27 20:16:55", "temperature_f": 64}
9: {"city": "NYC", "timestamp": "2026-08-27 20:16:56", "temperature_f": 72}
10: {"city": "Pittsburgh", "timestamp": "2026-08-27 20:16:56", "temperature_f": 64}
11: {"city": "Pittsburgh", "timestamp": "2026-08-27 20:16:57", "temperature_f": 64}
12: {"city": "Pittsburgh", "timestamp": "2026-08-27 20:16:58", "temperature_f": 64}

What does the offset refer to?
position of message in partition